# Environment Setup

## Installing Libraries

In [ ]:
%pip install -q segmentation-models-pytorch torchmetrics kornia

## Importing Libraries

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from pathlib import Path
import albumentations as A
from albumentations.pytorch import ToTensorV2
import kornia.augmentation as K
import matplotlib.pyplot as plt
from enum import Enum
from pprint import pprint
from functools import lru_cache
import segmentation_models_pytorch as smp
from tqdm.auto import tqdm
import torchmetrics

## Setting up wandb

In [ ]:
import wandb
from kaggle_secrets import UserSecretsClient

# Log in using the key from Kaggle Secrets
user_secrets = UserSecretsClient()
wandb_api_key = user_secrets.get_secret('WANDB_API_KEY')
wandb.login(key=wandb_api_key)

wandb_api = wandb.Api()

ENTITY = 'subhrabiswas023-emoteai'
PROJECT = 'rooftop-detection-semantic-segmentation'

## Device setup for GPU acceleration

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## Setting Seed for Reproducibility

In [ ]:
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

# Data Preparation

## Defining Color Scheme for the Labels

In [ ]:
class SegmentationClass(Enum):
    BACKGROUND = 0
    ROOFTOP = 1

## Prototyping Setup

In [ ]:
PROTOTYPING = True
PROTOTYPE_SIZE = 1
if PROTOTYPING:
    print("Prototyping mode is ON with size =", PROTOTYPE_SIZE)

## Creating Dataset Class

In [ ]:
from typing import Optional, Callable

COLOR_THRESHOLD = 128 # Median of 0 (Black) and 255 (White)

@lru_cache(maxsize=8)
def load_image(image_path: Path) -> np.ndarray:
    """
    Loads an image.
    Results are cached based on the input 'image_path'.
    """   
    return np.array(Image.open(image_path).convert("RGB"))


@lru_cache(maxsize=8)
def load_and_process_mask(mask_path: Path) -> np.ndarray:
    """
    Loads a mask.
    Results are cached based on the input 'mask_path'.
    """
    mask_gray = np.array(Image.open(mask_path).convert("L"))
    return (mask_gray > COLOR_THRESHOLD).astype(np.int64)  

class PatchedSegmentationDataset(Dataset):
    def __init__(self, image_dir: Path, mask_dir: Path, patch_size: int, transform: Optional[Callable] = None):
        """
        Args:
            image_dir (Path): Path to the images folder
            mask_dir (Path): Path to the masks folder
            patch_size (int): The size of the patches to extract
            transform (callable, optional): Albumentations transform
        """
        self.patch_size = patch_size
        self.transform = transform

        self.image_dir = image_dir
        self.mask_dir = mask_dir

        self.image_paths = sorted(self.image_dir.iterdir())
            
        self.patch_info = []
        for image_path in self.image_paths:
            with Image.open(image_path) as img:
                width, height = img.size

            n_patches_y = height // patch_size
            n_patches_x = width // patch_size

            for i in range(n_patches_y):
                for j in range(n_patches_x):
                    self.patch_info.append(
                        {
                            "image_path": image_path,
                            "patch_y_idx": i,
                            "patch_x_idx": j,
                        }
                    )

        if PROTOTYPING:
            self.patch_info = self.patch_info[:PROTOTYPE_SIZE]

    def __len__(self):
        return len(self.patch_info)

    def __getitem__(self, idx):
        info = self.patch_info[idx]
        
        image_path = info["image_path"]
        mask_path = self.mask_dir / image_path.name
        
        image = load_image(image_path)
        mask_class_indices = load_and_process_mask(mask_path)
        
        patch_y_idx = info["patch_y_idx"]
        patch_x_idx = info["patch_x_idx"]

        y_start = patch_y_idx * self.patch_size
        x_start = patch_x_idx * self.patch_size

        image_patch = image[y_start: y_start + self.patch_size, x_start: x_start + self.patch_size]
        mask_patch = mask_class_indices[y_start: y_start + self.patch_size, x_start: x_start + self.patch_size]

        if self.transform:
            augmented = self.transform(image=image_patch, mask=mask_patch)
            image_patch = augmented["image"]
            mask_patch = augmented["mask"]

        return image_patch, mask_patch.long()
        

## Preparing Pre-transformation

In [ ]:
CROP_SIZE = 256

pre_transform = A.Compose([
    A.Resize(CROP_SIZE, CROP_SIZE),
    ToTensorV2(),
])

## Creating The Dataset

In [ ]:
PATCH_SIZE = 256

root_dir = Path('/kaggle/input/inria-rooftop-segmentation-dataset-1024x1024-png')
train_dir = root_dir / 'Arial_images_1024_1024'
val_dir = root_dir / 'Arial_validation_images'

train_dataset = PatchedSegmentationDataset(train_dir / 'images', train_dir / 'masks', PATCH_SIZE, pre_transform)
val_dataset = PatchedSegmentationDataset(val_dir / 'images', val_dir / 'masks', PATCH_SIZE, pre_transform)

print(f"Train dataset: {len(train_dataset)} patches")
print(f"Validation dataset: {len(val_dataset)} patches")

## Exploring Few Samples from The Dataset

In [ ]:
num_samples = 5

def visualize(dataset, num_samples):
    random_indices = torch.randperm(len(dataset))[:num_samples]
    for idx in random_indices:
        image, mask = dataset[idx]
    
        image = image.detach().cpu().permute(1, 2, 0).float().numpy() / 255.0 # (C, H, W) -> (H, W, C) for plotting
        # image = (image * STD) + MEAN
        image = np.clip(image, 0, 1)
    
        mask = mask.detach().cpu().numpy()
    
        fig, ax = plt.subplots(1, 2, figsize=(10, 5))
        
        ax[0].imshow(image)
        ax[0].set_title(f"Image patch {idx}")
        ax[0].axis("off")
        
        ax[1].imshow(mask, cmap='gray')
        ax[1].set_title(f"Mask patch {idx}")
        ax[1].axis("off")
    
        plt.show()

visualize(train_dataset, num_samples)

## Preparaing Data Loader

In [ ]:
BATCH_SIZE = 16
NUM_WORKERS = 0

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

def build_loader(dataset: PatchedSegmentationDataset, batch_size: int, shuffle: bool=False):
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle, num_workers=NUM_WORKERS, pin_memory=True)

# Model Selection

## Defining The Hyperparameters

In [ ]:
# The hyperparameter search space
sweep_config = dict(
    method='grid',
    metric=dict(
        name='val_miou',
        goal='maximize'
    ),
    parameters=dict(
        learning_rate=dict(
            value=3e-4
        ),
        batch_size=dict(
            value=32
        ),
        encoder=dict(
            values=['mobilenet_v2', 'efficientnet-b0', 'resnet18']
        ),
        loss=dict(
            value='focal+dice'
        ),
        optimizer=dict(
            value='adam'
        ),
        loss_alpha=dict(
            # The weight for the Focal Loss part of the combined loss
            # A range of 0.3-0.7 ensures both losses contribute significantly.
            value=0.5
        ),
        epochs=dict(
            value=20
        )
    )
)

## Obtaining the sweep

In [ ]:
# Paste the sweep id to resume the unfinished runs of that sweep, keep to None for a new sweep
EXISTING_SWEEP_ID = None

if EXISTING_SWEEP_ID:
    # Use the same sweep
    print(f"[Recovery] Using existing sweep id: {EXISTING_SWEEP_ID}")
    sweep_id = EXISTING_SWEEP_ID
else:
    # Create a new sweep
    sweep_id = wandb.sweep(sweep_config, project=PROJECT, entity=ENTITY)

full_sweep_path = f"{ENTITY}/{PROJECT}/{sweep_id}"
sweep = wandb_api.sweep(full_sweep_path)

## Model Achitecture

In [ ]:
def build_model(encoder_name):
    model = smp.Unet(
        encoder_name=encoder_name,
        classes=len(SegmentationClass),
    )

    return model

## Preparing Transformation for Augmentation

In [ ]:
class SyncedImageMaskTransform(nn.Module):
    """Stacks image and mask tensor for performing random augmentation together, then returns the final image and mask"""
    def __init__(self, spatial_transform):
        super().__init__()
        self.spatial_transform = spatial_transform

    def forward(self, image, mask):
        mask = mask.unsqueeze(1).float()
        stacked = torch.cat([image, mask], dim=1)
        
        stacked = self.spatial_transform(stacked)
        
        image, mask = torch.split(stacked, [3, 1], dim=1)
        mask = mask.squeeze(1).long()
        
        return image, mask

spatial_transform = nn.Sequential(
    K.RandomHorizontalFlip(p=0.5),
    K.RandomVerticalFlip(p=0.5)
).to(device)

synced_transform = SyncedImageMaskTransform(spatial_transform).to(device)

## Normalizaiton Function

In [ ]:
MEAN = (0.485, 0.456, 0.406)
STD = (0.229, 0.224, 0.225)

normalize = K.Normalize(mean=torch.tensor(MEAN), std=torch.tensor(STD)).to(device)

## Creating Loss Function

In [ ]:
class CombinedLoss(nn.Module):
    '''Focal and dice loss added with the proportion of alpha'''
    def __init__(self, mode: str, alpha: float=0.5):
        super().__init__()
        self.alpha = alpha
        self.focal_loss_fn = smp.losses.FocalLoss(mode=mode)
        self.dice_loss_fn = smp.losses.DiceLoss(mode=mode)

    def forward(self, preds, targets):
        focal_loss = self.focal_loss_fn(preds, targets)
        dice_loss = self.dice_loss_fn(preds, targets)
        
        return self.alpha * focal_loss + (1 - self.alpha) * dice_loss

## Preparaing Optimizer

In [ ]:
def build_optimizer(model, optimizer: str, learning_rate: float):
    if optimizer == 'sgd':
        return optim.SGD(model.parameters(), lr=learning_rate, momentum=0.9)
    elif optimizer == 'adam':
        return optim.Adam(model.parameters(), lr=learning_rate)
    raise ValueError

# Training

## Single Training Epoch

In [ ]:
def train_epoch(model, train_loader, loss_fn, optimizer):
    model.train()
    running_train_loss = 0.0

    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        
        inputs = inputs.float() / 255.0
        inputs, labels = synced_transform(inputs, labels)
        inputs = normalize(inputs)
        
        outputs = model(inputs)
        
        loss = loss_fn(outputs, labels)
        running_train_loss += loss.item()
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    return running_train_loss / len(train_loader)

## Single Validation Epoch

In [ ]:
def val_epoch(model, val_loader, loss_fn):
    model.eval()
    running_val_loss = 0.0
    metric = torchmetrics.JaccardIndex(task='multiclass', num_classes=len(SegmentationClass)).to(device)
    
    with torch.inference_mode():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            
            inputs = inputs.float() / 255.0
            inputs = normalize(inputs)

            outputs = model(inputs)
            
            loss = loss_fn(outputs, labels)
            running_val_loss += loss.item()

            preds = torch.argmax(outputs, dim=1)
            metric.update(preds, labels)

        avg_val_loss = running_val_loss / len(val_loader)
        val_miou = metric.compute()

        metric.reset()

        return avg_val_loss, val_miou

## Training Loop

In [ ]:
def train(config=None, resume_id=None):
    # Initialize the run
    with wandb.init(config=config, save_code=True, id=resume_id, resume='allow') as run:
        # config = api_run.config if EXISTING_SWEEP_ID else wandb.config
        config = wandb.config

        # Name the run
        run.name = f'{config.encoder}_{config.learning_rate: .1e}_{run.id}'

        # Setup tags and add to the run
        tags = ['sweep', 'prototyping' if PROTOTYPING else 'full-dataset', config.encoder]
        run.tags = tags
        
        # Training variables
        start_epoch = 0
        
        train_loader = build_loader(train_dataset, config.batch_size, True)
        val_loader = build_loader(val_dataset, config.batch_size, False)
        
        model = build_model(config.encoder).to(device)
        
        loss_fn = CombinedLoss(mode='multiclass', alpha=config.loss_alpha).to(device)
        optimizer = build_optimizer(model, config.optimizer, config.learning_rate)
        
        best_val_miou = 0.0

        # Checkpoint file name
        checkpoint_path = 'checkpoint.pth'

        # Restore the checkpoint when resumed
        if run.resumed:
            try:
                # Retrieving the checkpoint
                checkpoint_file = run.restore(checkpoint_path)
                checkpoint = torch.load(checkpoint_file.name, map_location=device)

                # Retrieving the variables
                start_epoch = checkpoint['epoch'] + 1
                model.load_state_dict(checkpoint['model_state_dict'])
                optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
                best_val_miou = checkpoint['best_val_miou']

                print(f'Ready to resume from {start_epoch}')

            except Exception as e:
                # In that case, the save will not be available
                # So, we shall start from scratch
                print(f"Couldn't load the checkpoint. Starting from scratch. Error: {e}")
                

        # Training loop
        for epoch in range(start_epoch, config.epochs):
            # Train and validate, then get the metrics
            avg_train_loss = train_epoch(model, train_loader, loss_fn, optimizer)
            avg_val_loss, val_miou = val_epoch(model, val_loader, loss_fn)

            # Log the metrics
            run.log(dict(
                train_loss=avg_train_loss,
                val_loss=avg_val_loss,
                val_miou=val_miou
            ))

            # Prepare a checkpoint
            checkpoint = dict(
                epoch=epoch,
                model_state_dict=model.state_dict(),
                optimizer_state_dict=optimizer.state_dict(),
                best_val_miou=best_val_miou
            )
            
            # Save the checkpoint
            torch.save(checkpoint, checkpoint_path)
            wandb.save(checkpoint_path)

            # When the best_val_loss is beaten
            if val_miou > best_val_miou:
                # This is the best validation loss
                best_val_miou = val_miou

                # Best model file name
                best_model_path = 'best_model.pth'

                # Save the best model
                torch.save(model.state_dict(), best_model_path)
                wandb.save(best_model_path)

                print(f"Run {run.name}: New best model saved at epoch {epoch+1}")
                

## Resuming Crashed Runs

In [ ]:
# Check for unfinished runs in the sweep
# The sweep has already been created earlier
# if EXISTING_SWEEP_ID:
#     for run in sweep.runs:
#         # The crash is most likey because the kernel is stopped before completion
#         if run.state == 'crashed':
#             train(resume_id=run.id)

## Sweep Agent

In [ ]:
wandb.agent(full_sweep_path, train)

# Testing

## Obtaining The Best Run from The Sweep

In [ ]:
best_run = sweep.best_run()
print(f"Found best run: {best_run.name}")
print(f"Best mIoU: {best_run.summary['val_miou']:.4f}")

## Obtaining The Best Hyperparameters

In [ ]:
best_hyperparameters = best_run.config
print("Best Hyperparameters:")
pprint(best_hyperparameters)